# 工具
将函数公开为您的 MCP 客户端的可执行功能。

工具是核心构建块，它允许你的 LLM 与外部系统交互、执行代码以及访问其训练数据之外的数据。在 FastMCP 中，工具是通过 MCP 协议暴露给 LLM 的 Python 函数。

## 什么是工具？
FastMCP 中的工具将常规 Python 函数转换为 LLM 可以在对话过程中调用的功能。当 LLM 决定使用某个工具时：
 
1. 它根据工具的模式发送带有参数的请求。   
2. FastMCP 根据您的函数签名验证这些参数。   
3. 您的函数将使用经过验证的输入执行。   
4. 结果返回给 LLM，LLM 可以在其响应中使用它。  

这使得 LLM 能够执行查询数据库、调用 API、进行计算或访问文件等任务，从而将其功能扩展到训练数据之外。

## 工具
` @tool装饰者`
创建工具就像使用以下方法修饰 Python 函数一样简单`@mcp.tool()`：


In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="CalculatorServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Adds two integer numbers together."""
    return a + b

注册此工具后，FastMCP 会自动：

- 使用函数名称（add）作为工具名称。
- 使用函数的文档字符串（Adds two integer numbers...）作为工具描述。
- 根据函数的参数和类型注释生成输入模式。
- 处理参数验证和错误报告。

定义 Python 函数的方式决定了该工具在 LLM 客户端中的显示和行为方式。

> 注意：带有`*args`或`**kwarg`的函数s不支持作为工具。存在此限制是因为 FastMCP 需要为 MCP 协议生成完整的参数模式，而变量参数列表无法做到这一点。


### 参数
​
`注释`             
参数的类型注解对于工具的正常运行至关重要。它们：

- 告知 LLM 每个参数的预期数据类型
- 启用 FastMCP 来验证来自客户端的输入数据
- 为 MCP 协议生成准确的 JSON 模式
- 对参数使用标准 Python 类型注释：

In [ ]:
@mcp.tool()
def analyze_text(
    text: str,
    max_tokens: int = 100,
    language: str | None = None
) -> dict:
    """Analyze the provided text."""
    # Implementation...

### 参数元数据
您可以使用带有`Annotated`的Pydantic`Field`类提供有关参数的其他元数据。这种方法更受欢迎，因为它更现代，并且将类型提示与验证规则分开：


In [ ]:
from typing import Annotated
from pydantic import Field

@mcp.tool()
def process_image(
    image_url: Annotated[str, Field(description="URL of the image to process")],
    resize: Annotated[bool, Field(description="Whether to resize the image")] = False,
    width: Annotated[int, Field(description="Target width in pixels", ge=1, le=2000)] = 800,
    format: Annotated[
        Literal["jpeg", "png", "webp"], 
        Field(description="Output image format")
    ] = "jpeg"
) -> dict:
    """Process an image with optional resizing."""
    # Implementation...

您也可以使用字段作为默认值，尽管最好使用带注释的方法：

In [ ]:
@mcp.tool()
def search_database(
    query: str = Field(description="Search query string"),
    limit: int = Field(10, description="Maximum number of results", ge=1, le=100)
) -> list:
    """Search the database with the provided query."""
    # Implementation...

Field 提供了多种验证和文档功能：

- description：参数的人类可读解释（显示给 LLM）
- ge/ gt/ le/ lt：大于/小于（或等于）约束
- min_length/ max_length：字符串或集合长度限制
- pattern：用于字符串验证的正则表达式模式
- default：如果省略参数则使用默认值

### 支持的类型 

FastMCP 支持多种类型注释，包括所有 Pydantic 类型：


| Type Annotation | Example | Description |
| :--- | :--- | :--- |
| Basic types | int , float , str , bool | Simple scalar values - see Built-in Types |
| Binary data | bytes | Binary content - see Binary Data |
| Date and Time | datetime, date, timedelta | Date and time objects - see Date and Time Types |
| Collection types | list[str] , dict[str, int] , set[int] | Collections of items - see Collection Types |
| Optional types | float \| None, Optional[float] | Parameters that may be null/omitted - see Union and Optional Types |
| Union types | str \| int , Union[str, int] | Parameters accepting multiple types - see Union and Optional Types |
| Constrained types | Literal["A", "B"] , Enum | Parameters with specific allowed values - see Constrained Types |
| Paths | Path | File system paths - see Paths |
| UUIDs | UUID | Universally unique identifiers - see UUIDs |
| Pydantic models | UserData | Complex structured data - see Pydantic Models |

### 可选参数
FastMCP 遵循 Python 标准函数参数约定。没有默认值的参数是必需的，而有默认值的参数是可选的。

In [ ]:
@mcp.tool()
def search_products(
    query: str,                   # Required - no default value
    max_results: int = 10,        # Optional - has default value
    sort_by: str = "relevance",   # Optional - has default value
    category: str | None = None   # Optional - can be None
) -> list[dict]:
    """Search the product catalog."""
    # Implementation...

## 元数据
虽然 FastMCP 从您的函数中推断出名称和描述，但您可以覆盖这些并使用@mcp.tool装饰器的参数添加标签：

- name：设置通过 MCP 公开的明确工具名称。
- description：提供通过 MCP 公开的描述。如果设置，则函数的文档字符串将被忽略。
- tags：用于对工具进行分类的一组字符串。客户端可能会使用标签来筛选或分组可用的工具。

In [ ]:
@mcp.tool(
    name="find_products",           # Custom tool name for the LLM
    description="Search the product catalog with optional category filtering.", # Custom description
    tags={"catalog", "search"}      # Optional tags for organization/filtering
)
def search_products_implementation(query: str, category: str | None = None) -> list[dict]:
    """Internal function description (ignored if description is provided above)."""
    # Implementation...
    print(f"Searching for '{query}' in category '{category}'")
    return [{"id": 2, "name": "Another Product"}]

## 异步工具
FastMCP 无缝支持标准（def）和异步（async def）函数作为工具。


当工具需要执行可能需要等待外部系统（网络请求、数据库查询、文件访问）的操作时，请使用 `async def`，以保持服务器的响应速度。

In [ ]:
# Synchronous tool (suitable for CPU-bound or quick tasks)
@mcp.tool()
def calculate_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate the distance between two coordinates."""
    # Implementation...
    return 42.5

# Asynchronous tool (ideal for I/O-bound operations)
@mcp.tool()
async def fetch_weather(city: str) -> dict:
    """Retrieve current weather conditions for a city."""
    # Use 'async def' for operations involving network calls, file I/O, etc.
    # This prevents blocking the server while waiting for external operations.
    async with aiohttp.ClientSession() as session:
        async with session.get(f"https://api.example.com/weather/{city}") as response:
            # Check response status before returning
            response.raise_for_status()
            return await response.json()

## 返回值

FastMCP 会自动将你的函数返回的值转换为适合客户端的 MCP 内容格式：
- `str`：已发送TextContent。
- `dict`，，listPydanticBaseModel：序列化为 JSON 字符串并作为发送TextContent。
- `bytes`：Base64 编码并发送为BlobResourceContents（通常在内EmbeddedResource）。
- `fastmcp.Image`：用于轻松返回图像数据的辅助类。以 形式发送ImageContent。
- `None`：导致空响应（没有内容发送回客户端）。

如果可能的话，FastMCP 将尝试将其他类型序列化为字符串。

> 目前，FastMCP 仅响应您的工具的返回值，而不是其返回注释。

In [ ]:
from fastmcp import FastMCP, Image
import io
try:
    from PIL import Image as PILImage
except ImportError:
    raise ImportError("Please install the `pillow` library to run this example.")

mcp = FastMCP("Image Demo")

@mcp.tool()
def generate_image(width: int, height: int, color: str) -> Image:
    """Generates a solid color image."""
    # Create image using Pillow
    img = PILImage.new("RGB", (width, height), color=color)

    # Save to a bytes buffer
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    img_bytes = buffer.getvalue()

    # Return using FastMCP's Image helper
    return Image(data=img_bytes, format="png")

@mcp.tool()
def do_nothing() -> None:
    """This tool performs an action but returns no data."""
    print("Performing a side effect...")
    return None

## 错误处理

如果工具遇到错误，可以引发标准 Python 异常（`ValueError`、`TypeError`、`FileNotFoundError`、`自定义异常`等）或 FastMCP `ToolError`。


在所有情况下，异常都会被记录并转换为 MCP 错误响应，然后发送回客户端 LLM。出于安全原因，默认情况下，错误消息不包含在响应中。但是，如果您引发ToolError，则异常的内容将包含在响应中。这允许您选择向客户端 LLM 提供信息丰富的错误消息，以帮助 LLM 了解故障并做出适当的反应。

In [ ]:
from fastmcp import FastMCP
from fastmcp.exceptions import ToolError

@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b."""

    # Python exceptions raise errors but the contents are not sent to clients
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        raise TypeError("Both arguments must be numbers.")

    if b == 0:
        # ToolError contents are sent back to clients
        raise ToolError("Division by zero is not allowed.")
    return a / b

## 注释

FastMCP 允许您通过注释向工具添加专用元数据。这些注释可以传达工具对客户端应用程序的行为方式，而无需在 LLM 提示中使用 token 上下文。

注释在客户端应用程序中有多种用途：

- 添加用户友好的标题以便于显示
- 指示工具是否修改数据或系统
- 描述工具的安全性（破坏性与非破坏性）
- 如果工具与外部系统交互则发出信号

您可以使用 `@mcp.tool（）` 装饰器中的 `annotations` 参数向工具添加注释：

In [ ]:
@mcp.tool(
    annotations={
        "title": "Calculate Sum",
        "readOnlyHint": True,
        "openWorldHint": False
    }
)
def calculate_sum(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

FastMCP 支持以下标准注释：

| 注解 | 类型 | 默认 | 目的 |
| :--- | :--- | :--- | :--- |
| title | 细绳 | － | 用户界面的显示名称 |
| readOnlyHint | 布尔值 | 错误的 | 指示该工具是否只读取而不进行更改 |
| destructiveHint | 布尔值 | 真的 | 对于非只读工具，如果更改具有破坏性，则发出信号 |
| idempotentHint | 布尔值 | 错误的 | 指示重复相同的调用是否与单次调用具有相同的效果 |
| openWorldHint | 布尔值 | 真的 | 指定该工具是否与外部系统交互 |

请记住，注释有助于提升用户体验，但应将其视为建议性提示。它们可以帮助客户端应用程序呈现合适的 UI 元素和安全控制，但本身并不能强制执行安全边界。务必确保注释准确反映工具的实际功能。

## MCP 上下文
工具可以通过 Context 对象访问日志、读取资源或报告进度等 MCP 功能。要使用它，请在工具函数中添加一个类型提示为 Context 的参数。

In [ ]:
from fastmcp import FastMCP, Context

mcp = FastMCP(name="ContextDemo")

@mcp.tool()
async def process_data(data_uri: str, ctx: Context) -> dict:
    """Process data from a resource with progress reporting."""
    await ctx.info(f"Processing data from {data_uri}")
    
    # Read a resource
    resource = await ctx.read_resource(data_uri)
    data = resource[0].content if resource else ""
    
    # Report progress
    await ctx.report_progress(progress=50, total=100)
    
    # Example request to the client's LLM for help
    summary = await ctx.sample(f"Summarize this in 10 words: {data[:200]}")
    
    await ctx.report_progress(progress=100, total=100)
    return {
        "length": len(data),
        "summary": summary.text
    }

上下文对象提供以下访问权限

- 记录：`ctx.debug()`，`ctx.info()`，`ctx.warning()`，​`​ctx.error()`
进度报告：`ctx.report_progress(progress, total)`
资源访问：`ctx.read_resource(uri)`
LLM 抽样：`ctx.sample(...)`
索取资料：`ctx.request_id，ctx.client_id`

## 参数类型
FastMCP 支持多种参数类型，为您设计工具时提供灵活性。

FastMCP 通常支持 Pydantic 支持的所有字段类型，包括所有 Pydantic 自定义类型。这意味着您可以在工具参数中使用任何 Pydantic 能够验证和解析的类型。

FastMCP 在可能的情况下支持类型强制转换。这意味着，如果客户端发送的数据与预期类型不匹配，FastMCP 将尝试将其转换为合适的类型。例如，如果客户端为注释为 int 的参数发送字符串，FastMCP 将尝试将其转换为整数。如果无法转换，FastMCP 将返回验证错误。


#### 内置类型
最常见的参数类型是 Python 的内置标量类型：

In [ ]:
@mcp.tool()
def process_values(
    name: str,             # Text data
    count: int,            # Integer numbers
    amount: float,         # Floating point numbers
    enabled: bool          # Boolean values (True/False)
):
    """Process various value types."""
    # Implementation...

这些类型为 LLM 提供了明确的预期，即哪些值是可接受的，并允许 FastMCP 正确验证输入。即使客户端提供了像“42”这样的字符串，对于标注为 int的参数，它也会被强制转换为整数。

#### 日期和时间类型
FastMCP 支持datetime模块中的各种日期和时间类型：

In [ ]:
from datetime import datetime, date, timedelta

@mcp.tool()
def process_date_time(
    event_date: date,             # ISO format date string or date object
    event_time: datetime,         # ISO format datetime string or datetime object
    duration: timedelta = timedelta(hours=1)  # Integer seconds or timedelta
) -> str:
    """Process date and time information."""
    # Types are automatically converted from strings
    assert isinstance(event_date, date)  
    assert isinstance(event_time, datetime)
    assert isinstance(duration, timedelta)
    
    return f"Event on {event_date} at {event_time} for {duration}"

#### 集合类型
FastMCP 支持所有标准 Python 集合类型：


In [ ]:
@mcp.tool()
def analyze_data(
    values: list[float],           # List of numbers
    properties: dict[str, str],    # Dictionary with string keys and values
    unique_ids: set[int],          # Set of unique integers
    coordinates: tuple[float, float],  # Tuple with fixed structure
    mixed_data: dict[str, list[int]] # Nested collections
):
    """Analyze collections of data."""
    # Implementation...

所有集合类型都可以用作参数注释：

- `list[T]`- 物品的有序序列
- `dict[K, V]`- 键值映射
- `set[T]`- 无序收集独特物品
- `tuple[T1, T2, ...]`- 固定长度序列，可能具有不同的类型

集合类型可以嵌套和组合，以表示复杂的数据结构。符合预期结构的 JSON 字符串将被自动解析并转换为适当的 Python 集合类型。

#### 联合类型和可选类型

对于可以接受多种类型或可以省略的参数：

现代 Python 语法 ( str | int) 优于旧Union[str, int]形式。同样，str | None优于Optional[str]。

In [ ]:
@mcp.tool()
def flexible_search(
    query: str | int,              # Can be either string or integer
    filters: dict[str, str] | None = None,  # Optional dictionary
    sort_field: str | None = None  # Optional string
):
    """Search with flexible parameter types."""
    # Implementation...

#### 约束类型
当参数必须是一组预定义值之一时，您可以使用文字类型或枚举：

##### 文字
文字将参数限制为特定的一组值：

文字类型：

- 直接在类型注释中指定精确的允许值
- 帮助LLM准确理解哪些价值观是可以接受的
- 提供输入验证（无效值错误）
- 为客户创建清晰的架构

In [ ]:
from typing import Literal

@mcp.tool()
def sort_data(
    data: list[float],
    order: Literal["ascending", "descending"] = "ascending",
    algorithm: Literal["quicksort", "mergesort", "heapsort"] = "quicksort"
):
    """Sort data using specific options."""
    # Implementation...

##### 枚举
对于更结构化的约束值集，请使用 Python 的 Enum 类：

In [ ]:
from enum import Enum

class Color(Enum):
    RED = "red"
    GREEN = "green"
    BLUE = "blue"

@mcp.tool()
def process_image(
    image_path: str, 
    color_filter: Color = Color.RED
):
    """Process an image with a color filter."""
    # Implementation...
    # color_filter will be a Color enum member

使用枚举类型时：

- 客户端应该提供枚举的值（例如“red”），而不是枚举成员名称（例如“RED”）
- FastMCP 自动将字符串值强制转换为适当的 Enum 对象
- 您的函数接收实际的 Enum 成员（例如Color.RED）
- 对于不在枚举中的值，将引发验证错误

##### 二进制数据
处理工具参数中的二进制数据有两种方法：
##### 字节

In [ ]:
@mcp.tool()
def process_binary(data: bytes):
    """Process binary data directly.
    
    The client can send a binary string, which will be 
    converted directly to bytes.
    """
    # Implementation using binary data
    data_length = len(data)
    # ...

#### Base64 编码字符串

当您希望从客户端接收 base64 编码的二进制数据时，建议使用此方法。

​


In [ ]:
from typing import Annotated
from pydantic import Field

@mcp.tool()
def process_image_data(
    image_data: Annotated[str, Field(description="Base64-encoded image data")]
):
    """Process an image from base64-encoded string.
    
    The client is expected to provide base64-encoded data as a string.
    You'll need to decode it manually.
    """
    # Manual base64 decoding
    import base64
    binary_data = base64.b64decode(image_data)
    # Process binary_data...

#### 路径
Path模块的类型可pathlib用于文件系统路径：


In [ ]:
from pathlib import Path

@mcp.tool()
def process_file(path: Path) -> str:
    """Process a file at the given path."""
    assert isinstance(path, Path)  # Path is properly converted
    return f"Processing file at {path}"

#### UUID
UUID模块中的类型可uuid用于唯一标识符：

当客户端发送字符串 UUID（例如“123e4567-e89b-12d3-a456-426614174000”）时，FastMCP 会自动将其转换为UUID对象。

In [ ]:
import uuid

@mcp.tool()
def process_item(
    item_id: uuid.UUID  # String UUID or UUID object
) -> str:
    """Process an item with the given UUID."""
    assert isinstance(item_id, uuid.UUID)  # Properly converted to UUID
    return f"Processing item {item_id}"

#### Pydantic 模型
对于具有嵌套字段和验证的复杂结构化数据，请使用 Pydantic 模型：

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class User(BaseModel):
    username: str
    email: str = Field(description="User's email address")
    age: int | None = None
    is_active: bool = True

@mcp.tool()
def create_user(user: User):
    """Create a new user in the system."""
    # The input is automatically validated against the User model
    # Even if provided as a JSON string or dict
    # Implementation...

使用 Pydantic 模型可以提供：

- 清晰、自文档化的结构，适用于复杂的输入
- 内置数据验证
- 自动生成 LLM 的详细 JSON 模式
- 从 dict/JSON 输入自动转换

客户端可以通过以下方式提供 Pydantic 模型参数的数据：

- JSON 对象（字符串）
- 具有适当结构的字典
- 适当格式的嵌套参数

#### Pydantic 字段
FastMCP 通过 Pydantic 的类支持强大的参数验证Field。这对于确保输入值满足除其类型之外的特定要求特别有用。

> 请注意，可以在`Pydantic`模型之外使用字段来提供元数据和验证约束。 首选的方法是使用带有`Annotatedwith`的`Field`


In [ ]:
from typing import Annotated
from pydantic import Field

@mcp.tool()
def analyze_metrics(
    # Numbers with range constraints
    count: Annotated[int, Field(ge=0, le=100)],         # 0 <= count <= 100
    ratio: Annotated[float, Field(gt=0, lt=1.0)],       # 0 < ratio < 1.0
    
    # String with pattern and length constraints
    user_id: Annotated[str, Field(
        pattern=r"^[A-Z]{2}\d{4}$",                     # Must match regex pattern
        description="User ID in format XX0000"
    )],
    
    # String with length constraints
    comment: Annotated[str, Field(min_length=3, max_length=500)] = "",
    
    # Numeric constraints
    factor: Annotated[int, Field(multiple_of=5)] = 10,  # Must be multiple of 5
):
    """Analyze metrics with validated parameters."""
    # Implementation...

您也可以使用`Field`默认值，尽管这种`Annotated`方法是首选：

In [ ]:
@mcp.tool()
def validate_data(
    # Value constraints
    age: int = Field(ge=0, lt=120),                     # 0 <= age < 120
    
    # String constraints
    email: str = Field(pattern=r"^[\w\.-]+@[\w\.-]+\.\w+$"),  # Email pattern
    
    # Collection constraints
    tags: list[str] = Field(min_length=1, max_length=10)  # 1-10 tags
):
    """Process data with field validations."""
    # Implementation...

常见的验证选项包括：

| 验证 |  | 类型 | 描述 |
| :--- | :--- | :--- | :--- |
| ge，gt |  | 数字 | 大于（或等于）约束 |
| le，lt |  | 数字 | 小于（或等于）约束 |
| multiple＿of |  | 数字 | 值必须是该数字的倍数 |
| min＿length ， | max＿length | 字符串、列表等。 | 长度限制 |
| pattern |  | 细绳 | 正则表达式模式约束 |
| description |  | 任何 | 人类可读的描述（出现在模式中） |

当客户端发送无效数据时，FastMCP 将返回验证错误，解释参数验证失败的原因。


## 服务器行为
​
### 复制工具 New in version: 2.1.0

您可以控制 FastMCP 服务器在尝试注册多个同名工具时的行为。在创建 FastMCP 实例时，可使用 `on_duplicate_tools` 参数对此进行配置。


重复行为选项包括：

- "warn"（默认）：记录警告并且新工具替换旧工具。
- "error"：提出ValueError，防止重复注册。
- "replace"：默默地用新工具替换现有工具。
- "ignore"：保留原始工具并忽略新的注册尝试。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="StrictServer",
    # Configure behavior for duplicate tool names
    on_duplicate_tools="error"
)

@mcp.tool()
def my_tool(): return "Version 1"

# This will now raise a ValueError because 'my_tool' already exists
# and on_duplicate_tools is set to "error".
# @mcp.tool()
# def my_tool(): return "Version 2"

### 移除工具New in version: 2.3.4

您可以使用remove_tool方法从服务器动态删除工具：

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="DynamicToolServer")

@mcp.tool()
def calculate_sum(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

mcp.remove_tool("calculate_sum")

### 旧版 JSON 解析  New in version: 2.2.10

FastMCP 1.0 和低于 2.2.10 的版本依赖于一种尝试绕过 LLM 限制的方法，即自动解析工具参数中的字符串化 JSON（例如，转换"[1,2,3]"为[1,2,3]）。从 FastMCP 2.2.10 开始，此行为默认被禁用，因为它会绕过类型验证，并可能导致意外的类型强制转换问题（例如，将“true”解析为布尔值，并尝试调用一个需要字符串的工具，这将导致类型验证失败）。

大多数现代 LLM 都正确格式化了 JSON，但如果使用不必要地将 JSON 字符串化的模型（就像 2024 年末的 Claude Desktop 的情况一样），则可以通过设置环境变量在服务器上重新启用此行为FASTMCP_TOOL_ATTEMPT_PARSE_JSON_ARGS=1。